In [0]:
--SELECT version();
--USE DATABASE qdp_database;

CREATE SCHEMA IF NOT EXISTS qdp_episodes_of_care;
SET search_path TO qdp_episodes_of_care;


-- metadata table
DROP TABLE IF EXISTS metadata;


CREATE TABLE metadata (
	supplier VARCHAR,
	data_product VARCHAR,
	data_product_version VARCHAR,
	schema_name VARCHAR,
	schema_version VARCHAR,
	schema_variant_name VARCHAR,
	schema_variant_version VARCHAR,
	instance VARCHAR,
	instance_version VARCHAR
 );



INSERT INTO metadata
(
  supplier,
  data_product,
  data_product_version,
  schema_name,
  schema_version,
  schema_variant_name,
  schema_variant_version,
  instance,
  instance_version
)
VALUES (
  'quantexa.com',
  'com.quantexa.qdp.episodes_of_care',
  '0.1',
  'episodes_of_care.data_vault',
  '0.1',
  'base',
  '0.1',
  'not yet populated with data',
  ''
);




-- -----------------------------------------------------------
-- create hub_episode_of_care table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS hub_episode_of_care CASCADE;

CREATE TABLE hub_episode_of_care (
    episode_of_care_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    episode_of_care_entity_id VARCHAR,
  	tennant_id BIGINT NOT NULL DEFAULT 0,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE hub_episode_of_care ADD CONSTRAINT pk_episodeofcareall_hubepisodeofcare_episodeofcareid PRIMARY KEY (episode_of_care_id);

COMMENT ON COLUMN hub_episode_of_care.episode_of_care_id IS 'Episode Of Care Identity';




-- -----------------------------------------------------------
-- create sat_episode_of_care table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_episode_of_care;

CREATE TABLE sat_episode_of_care (
    sat_episode_of_care_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    episode_of_care_id BIGINT NOT NULL,
    load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    record_source_id BIGINT NOT NULL DEFAULT 1,
	data_source_code VARCHAR,
	data_source_concept_id BIGINT,
	data_source_raw_code VARCHAR,
	data_source_raw_concept_id BIGINT,
	encounter_id BIGINT,
	patient_party_id BIGINT,
	managing_organisation_party_id BIGINT,
	care_manager_party_id INT,
	status_code VARCHAR,
	status_concept_id BIGINT,
	status_raw_code VARCHAR,
	status_raw_concept_id BIGINT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_episode_of_care ADD CONSTRAINT pk_episodeofcareall_satepisodeofcare_satepisodeofcareid PRIMARY KEY (sat_episode_of_care_id);
ALTER TABLE sat_episode_of_care ADD CONSTRAINT fk_episodeofcareall_satepisodeofcare_hubepisodeofcare_episodeofcareid FOREIGN KEY (episode_of_care_id) REFERENCES hub_episode_of_care(episode_of_care_id);

COMMENT ON COLUMN sat_episode_of_care.episode_of_care_id IS 'Episode Of Care Identity';
COMMENT ON COLUMN sat_episode_of_care.sat_episode_of_care_id IS 'Episode Of Care Identity Sat';
COMMENT ON COLUMN sat_episode_of_care.encounter_id IS 'The unique Patient Encounter Identifier for this Episode Of Care';
COMMENT ON COLUMN sat_episode_of_care.patient_party_id IS 'Unique identifier of the Patient who is the focus of this episode of care';
COMMENT ON COLUMN sat_episode_of_care.managing_organisation_party_id IS 'Unique identifier of the Organization that assumes responsibility for care coordination';
COMMENT ON COLUMN sat_episode_of_care.care_manager_party_id IS 'Care manager/care coordinator for the patient. See https://build.fhir.org/episodeofcare-definitions.html#EpisodeOfCare.careManager';
COMMENT ON COLUMN sat_episode_of_care.status_code IS 'The current Status of this Episode Of Care. E.g. Planned, Waitlist, Active, On Hold, Finished, Cancelled, Entered in Error. See http://hl7.org/fhir/episode-of-care-status';



-- -----------------------------------------------------------
-- create sat_episode_of_care_diagnosis table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_episode_of_care_diagnosis;

CREATE TABLE sat_episode_of_care_diagnosis (
	episode_of_care_diagnosis_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	episode_of_care_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	use_code VARCHAR,
	use_concept_id BIGINT,
	use_raw_code VARCHAR,
	use_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_episode_of_care_diagnosis ADD CONSTRAINT pk_episodeofcareall_satepisodeofcarediagnosis_episodeofcarediagnosisid PRIMARY KEY (episode_of_care_diagnosis_id);
ALTER TABLE sat_episode_of_care_diagnosis ADD CONSTRAINT fk_episodeofcareall_satepisodeofcarediagnosis_hubepisodeofcare_episodeofcareid FOREIGN KEY (episode_of_care_id) REFERENCES hub_episode_of_care(episode_of_care_id);

COMMENT ON COLUMN sat_episode_of_care_diagnosis.episode_of_care_id IS 'Episode Of Care Identity';
COMMENT ON COLUMN sat_episode_of_care_diagnosis.episode_of_care_diagnosis_id IS 'Episode Of Care Diagnosis Identity';
COMMENT ON COLUMN sat_episode_of_care_diagnosis.use_code IS 'Role that this diagnosis has within the episode of care. E.g. admission, billing, discharge. See https://build.fhir.org/episodeofcare-definitions.html#EpisodeOfCare.diagnosis';
COMMENT ON COLUMN sat_episode_of_care_diagnosis.rank IS 'The recommended order the Episode Of Care Diagnosis Use Codes should be presented';



-- -----------------------------------------------------------
-- create sat_episode_of_care_diagnosis_condition table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_episode_of_care_diagnosis_condition;

CREATE TABLE sat_episode_of_care_diagnosis_condition (
	episode_of_care_diagnosis_condition_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	episode_of_care_diagnosis_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	condition_code VARCHAR,
	condition_concept_id BIGINT,
	condition_raw_code VARCHAR,
	condition_raw_concept_id BIGINT,
	value VARCHAR,
	condition_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_episode_of_care_diagnosis_condition ADD CONSTRAINT pk_episodeofcareall_satepisodeofcarediagnosiscondition_episodeofcarediagnosisconditionid PRIMARY KEY (episode_of_care_diagnosis_condition_id);
ALTER TABLE sat_episode_of_care_diagnosis_condition ADD CONSTRAINT fk_episodeofcareall_satepisodeofcarediagnosiscondition_satepisodeofcarediagnosis_episodeofcarediagnosisid FOREIGN KEY (episode_of_care_diagnosis_id) REFERENCES sat_episode_of_care_diagnosis(episode_of_care_diagnosis_id);

COMMENT ON COLUMN sat_episode_of_care_diagnosis_condition.episode_of_care_diagnosis_id IS 'Episode Of Care Diagnosis Identity';
COMMENT ON COLUMN sat_episode_of_care_diagnosis_condition.episode_of_care_diagnosis_condition_id IS 'Episode Of Care Diagnosis Condition Identity';
COMMENT ON COLUMN sat_episode_of_care_diagnosis_condition.condition_code IS 'Episode of Care Diagnosis Condition Code. See https://build.fhir.org/condition.html';
COMMENT ON COLUMN sat_episode_of_care_diagnosis_condition.value IS 'Text Description of the Episode of Care Diagnosis Condition';
COMMENT ON COLUMN sat_episode_of_care_diagnosis_condition.condition_id IS 'Unique Identifier Reference to related Diagnosis Condition, if applicable';
COMMENT ON COLUMN sat_episode_of_care_diagnosis_condition.rank IS 'The recommended order the Episode Of Care Diagnosis Condition Codes should be presented';



-- -----------------------------------------------------------
-- create sat_episode_of_care_reason table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_episode_of_care_reason;

CREATE TABLE sat_episode_of_care_reason (
	episode_of_care_reason_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	episode_of_care_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	use_code VARCHAR,
	use_concept_id BIGINT,
	use_raw_code VARCHAR,
	use_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_episode_of_care_reason ADD CONSTRAINT pk_episodeofcareall_satepisodeofcarereason_episodeofcarereasonid PRIMARY KEY (episode_of_care_reason_id);
ALTER TABLE sat_episode_of_care_reason ADD CONSTRAINT fk_episodeofcareall_satepisodeofcarereason_hubepisodeofcare_episodeofcareid FOREIGN KEY (episode_of_care_id) REFERENCES hub_episode_of_care(episode_of_care_id);

COMMENT ON COLUMN sat_episode_of_care_reason.episode_of_care_id IS 'Episode Of Care Identity';
COMMENT ON COLUMN sat_episode_of_care_reason.episode_of_care_reason_id IS 'Episode Of Care Reason Identity';
COMMENT ON COLUMN sat_episode_of_care_reason.use_code IS 'Episode of Care Reason Use. E.g. CC (Chief Complaint), HC (Health Concern), AD (Admitting Diagnosis), RV (Reason for Visit), HM (Health Maintenance). See http://hl7.org/fhir/encounter-reason-use';
COMMENT ON COLUMN sat_episode_of_care_reason.rank IS 'The recommended order the Episode Of Care Reason Use Codes should be presented';



-- -----------------------------------------------------------
-- create sat_episode_of_care_reason_value table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_episode_of_care_reason_value;

CREATE TABLE sat_episode_of_care_reason_value (
	episode_of_care_reason_value_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	episode_of_care_reason_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	reason_code VARCHAR,
	reason_concept_id BIGINT,
	reason_raw_code VARCHAR,
	reason_raw_concept_id BIGINT,
	value VARCHAR,
	condition_id BIGINT,
	procedure_id BIGINT,
	observation_id BIGINT,
	healthcare_service_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_episode_of_care_reason_value ADD CONSTRAINT pk_episodeofcareall_satepisodeofcarereasonvalue_episodeofcarereasonvalueid PRIMARY KEY (episode_of_care_reason_value_id);
ALTER TABLE sat_episode_of_care_reason_value ADD CONSTRAINT fk_episodeofcareall_satepisodeofcarereasonvalue_satepisodeofcarereason_episodeofcarereasonid FOREIGN KEY (episode_of_care_reason_id) REFERENCES sat_episode_of_care_reason(episode_of_care_reason_id);

COMMENT ON COLUMN sat_episode_of_care_reason_value.episode_of_care_reason_id IS 'Episode Of Care Reason Identity';
COMMENT ON COLUMN sat_episode_of_care_reason_value.episode_of_care_reason_value_id IS 'Episode Of Care Reason Value Identity';
COMMENT ON COLUMN sat_episode_of_care_reason_value.reason_code IS 'Episode of Care Reason Code. See https://build.fhir.org/episodeofcare-definitions.html#EpisodeOfCare.reason.value';
COMMENT ON COLUMN sat_episode_of_care_reason_value.value IS 'Text Description of the Episode of Care Reason';
COMMENT ON COLUMN sat_episode_of_care_reason_value.condition_id IS 'Unique Identifier Reference to related Condition, if applicable';
COMMENT ON COLUMN sat_episode_of_care_reason_value.procedure_id IS 'Unique Identifier Reference to related Procedure, if applicable';
COMMENT ON COLUMN sat_episode_of_care_reason_value.observation_id IS 'Unique Identifier Reference to related Observation, if applicable';
COMMENT ON COLUMN sat_episode_of_care_reason_value.healthcare_service_id IS 'Unique Identifier Reference to related Healthcare Service, if applicable';
COMMENT ON COLUMN sat_episode_of_care_reason_value.rank IS 'The recommended order the Episode Of Care Reason Value Codes should be presented';



-- -----------------------------------------------------------
-- create sat_episode_of_care_referral_request table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_episode_of_care_referral_request;

CREATE TABLE sat_episode_of_care_referral_request (
	episode_of_care_referral_request_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	episode_of_care_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	service_request_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_episode_of_care_referral_request ADD CONSTRAINT pk_episodeofcareall_satepisodeofcarereferralrequest_episodeofcarereferralrequestid PRIMARY KEY (episode_of_care_referral_request_id);
ALTER TABLE sat_episode_of_care_referral_request ADD CONSTRAINT fk_episodeofcareall_satepisodeofcarereferralrequest_hubepisodeofcare_episodeofcareid FOREIGN KEY (episode_of_care_id) REFERENCES hub_episode_of_care(episode_of_care_id);

COMMENT ON COLUMN sat_episode_of_care_referral_request.episode_of_care_id IS 'Episode Of Care Identity';
COMMENT ON COLUMN sat_episode_of_care_referral_request.episode_of_care_referral_request_id IS 'Episode Of Care Referral Request Identity';
COMMENT ON COLUMN sat_episode_of_care_referral_request.service_request_id IS 'Unique Identifier for the originating Service Request. See https://build.fhir.org/episodeofcare-definitions.html#EpisodeOfCare.referralRequest';
COMMENT ON COLUMN sat_episode_of_care_referral_request.rank IS 'The recommended order the Episode Of Care Referral Requests should be presented';



-- -----------------------------------------------------------
-- create sat_episode_of_care_status_history table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_episode_of_care_status_history;

CREATE TABLE sat_episode_of_care_status_history (
	episode_of_care_status_history_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	episode_of_care_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	status_code VARCHAR,
	status_concept_id BIGINT,
	status_raw_code VARCHAR,
	status_raw_concept_id BIGINT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_episode_of_care_status_history ADD CONSTRAINT pk_episodeofcareall_satepisodeofcarestatushistory_episodeofcarestatushistoryid PRIMARY KEY (episode_of_care_status_history_id);
ALTER TABLE sat_episode_of_care_status_history ADD CONSTRAINT fk_episodeofcareall_satepisodeofcarestatushistory_hubepisodeofcare_episodeofcareid FOREIGN KEY (episode_of_care_id) REFERENCES hub_episode_of_care(episode_of_care_id);

COMMENT ON COLUMN sat_episode_of_care_status_history.episode_of_care_id IS 'Episode Of Care Identity';
COMMENT ON COLUMN sat_episode_of_care_status_history.episode_of_care_status_history_id IS 'Episode Of Care Status History Identity';
COMMENT ON COLUMN sat_episode_of_care_status_history.status_code IS 'The status of the Episode Of Care at this point in the history. E.g. Planned, Waitlist, Active, On Hold, Finished, Cancelled, Entered in Error. See http://hl7.org/fhir/episode-of-care-status';
COMMENT ON COLUMN sat_episode_of_care_status_history.period_start IS 'The start date and time of the period during which the EpisodeOfCare had this status. See https://build.fhir.org/episodeofcare-definitions.html#EpisodeOfCare.statusHistory.period';
COMMENT ON COLUMN sat_episode_of_care_status_history.period_end IS 'The end date and time of the period during which the EpisodeOfCare had this status. See https://build.fhir.org/episodeofcare-definitions.html#EpisodeOfCare.statusHistory.period';



-- -----------------------------------------------------------
-- create sat_episode_of_care_team table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_episode_of_care_team;

CREATE TABLE sat_episode_of_care_team (
	episode_of_care_team_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	episode_of_care_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	care_team_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_episode_of_care_team ADD CONSTRAINT pk_episodeofcareall_satepisodeofcareteam_episodeofcareteamid PRIMARY KEY (episode_of_care_team_id);
ALTER TABLE sat_episode_of_care_team ADD CONSTRAINT fk_episodeofcareall_satepisodeofcareteam_hubepisodeofcare_episodeofcareid FOREIGN KEY (episode_of_care_id) REFERENCES hub_episode_of_care(episode_of_care_id);

COMMENT ON COLUMN sat_episode_of_care_team.episode_of_care_id IS 'Episode Of Care Identity';
COMMENT ON COLUMN sat_episode_of_care_team.episode_of_care_team_id IS 'Episode Of Care Team Identity';
COMMENT ON COLUMN sat_episode_of_care_team.care_team_id IS 'The Care Team allocated to facilitate this episode of care. See https://build.fhir.org/episodeofcare-definitions.html#EpisodeOfCare.careTeam';
COMMENT ON COLUMN sat_episode_of_care_team.rank IS 'The recommended order the Episode Of Care Care Teams should be presented';



-- -----------------------------------------------------------
-- create sat_episode_of_care_type table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_episode_of_care_type;

CREATE TABLE sat_episode_of_care_type (
	episode_of_care_type_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	episode_of_care_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	type_code VARCHAR,
	type_concept_id BIGINT,
	type_raw_code VARCHAR,
	type_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_episode_of_care_type ADD CONSTRAINT pk_episodeofcareall_satepisodeofcaretype_episodeofcaretypeid PRIMARY KEY (episode_of_care_type_id);
ALTER TABLE sat_episode_of_care_type ADD CONSTRAINT fk_episodeofcareall_satepisodeofcaretype_hubepisodeofcare_episodeofcareid FOREIGN KEY (episode_of_care_id) REFERENCES hub_episode_of_care(episode_of_care_id);

COMMENT ON COLUMN sat_episode_of_care_type.episode_of_care_id IS 'Episode Of Care Identity';
COMMENT ON COLUMN sat_episode_of_care_type.episode_of_care_type_id IS 'Episode Of Care Type Identity';
COMMENT ON COLUMN sat_episode_of_care_type.type_code IS 'Type/class of the episode of care. E.g. hacc (Home and Community Care), pac (Post Acute Care), diab (Post coordinated diabetes program), da (Drug and alcohol rehabilitation), cacp (Community-based aged care). See http://terminology.hl7.org/CodeSystem/episodeofcare-type';
COMMENT ON COLUMN sat_episode_of_care_type.rank IS 'The recommended order the Episode Of Care Type Codes should be presented';



-- -----------------------------------------------------------
-- create view_episodes_of_care view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_episodes_of_care AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.episode_of_care_id,
	h.episode_of_care_entity_id,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
  s.sat_episode_of_care_id,
  s.encounter_id,
  s.patient_party_id,
  s.managing_organisation_party_id,
  s.care_manager_party_id,
  s.status_code,
  s.status_concept_id,
  s.status_raw_code,
  s.status_raw_concept_id,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_episodes_of_care.hub_episode_of_care h
    JOIN budd.qdp_episodes_of_care.sat_episode_of_care s
      ON h.episode_of_care_id = s.episode_of_care_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_episodes_of_care;


-- -----------------------------------------------------------
-- create view_episode_of_care_diagnoses view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_episode_of_care_diagnoses AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.episode_of_care_id,
	h.episode_of_care_entity_id,
  s.status_code AS episode_of_care_status_code,
  s.encounter_id,
  s.patient_party_id,
	ed.episode_of_care_diagnosis_id,
	ed.use_code AS diagnosis_use_code,
	ed.use_concept_id AS diagnosis_use_concept_id,
	ed.use_raw_code AS diagnosis_use_raw_code,
	ed.use_raw_concept_id AS diagnosis_use_raw_concept_id,
	ed.rank AS diagnosis_rank,
	ed.period_start AS diagnosis_period_start,
	ed.period_end AS diagnosis_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_episodes_of_care.hub_episode_of_care h
    JOIN budd.qdp_episodes_of_care.sat_episode_of_care s
      ON h.episode_of_care_id = s.episode_of_care_id
    JOIN budd.qdp_episodes_of_care.sat_episode_of_care_diagnosis ed
      ON h.episode_of_care_id = ed.episode_of_care_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_episode_of_care_diagnoses;


-- -----------------------------------------------------------
-- create view_episode_of_care_reasons view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_episode_of_care_reasons AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.episode_of_care_id,
	h.episode_of_care_entity_id,
  s.status_code AS episode_of_care_status_code,
  s.encounter_id,
  s.patient_party_id,
	er.episode_of_care_reason_id,
	er.use_code AS reason_use_code,
	er.use_concept_id AS reason_use_concept_id,
	er.use_raw_code AS reason_use_raw_code,
	er.use_raw_concept_id AS reason_use_raw_concept_id,
	er.rank AS reason_rank,
	er.period_start AS reason_period_start,
	er.period_end AS reason_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_episodes_of_care.hub_episode_of_care h
    JOIN budd.qdp_episodes_of_care.sat_episode_of_care s
      ON h.episode_of_care_id = s.episode_of_care_id
    JOIN budd.qdp_episodes_of_care.sat_episode_of_care_reason er
      ON h.episode_of_care_id = er.episode_of_care_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_episode_of_care_reasons;


-- -----------------------------------------------------------
-- create view_episode_of_care_status_history view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_episode_of_care_status_history AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.episode_of_care_id,
	h.episode_of_care_entity_id,
  s.status_code AS current_status_code,
  s.encounter_id,
  s.patient_party_id,
	esh.episode_of_care_status_history_id,
	esh.status_code AS history_status_code,
	esh.status_concept_id AS history_status_concept_id,
	esh.status_raw_code AS history_status_raw_code,
	esh.status_raw_concept_id AS history_status_raw_concept_id,
	esh.period_start AS history_period_start,
	esh.period_end AS history_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_episodes_of_care.hub_episode_of_care h
    JOIN budd.qdp_episodes_of_care.sat_episode_of_care s
      ON h.episode_of_care_id = s.episode_of_care_id
    JOIN budd.qdp_episodes_of_care.sat_episode_of_care_status_history esh
      ON h.episode_of_care_id = esh.episode_of_care_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_episode_of_care_status_history;


/**

**/
